# Retrieval at Scale — When Brute Force Breaks Down
## NI-ADM | Czech Technical University in Prague

In the previous tutorials we worked with ~20K candidate queries. A simple `np.argsort(scores)` runs in <1ms. But what happens when you have **1 million** items? **10 million**? **1 billion**?

This is the fundamental challenge of **large-scale retrieval**: the smartest model is useless if it's too slow.

We will:
1. **Feel the pain**: brute-force search at increasing scale (10K → 100K → 1M → 5M)
2. **Learn ANN indexing**: FAISS index types (Flat, IVF, IVF-PQ, HNSW)
3. **Measure the tradeoff**: plot Recall@K vs Queries Per Second
4. **See the cascade**: why production systems use retrieval + re-ranking

---

### The dilemma

| Approach | Accuracy | Speed at 1M items | Speed at 100M items |
|----------|----------|--------------------|-----------------|
| Brute force (exact) | Perfect | ~50ms | ~5 seconds |
| ANN (approximate) | ~95% recall | ~0.5ms | ~5ms |
| Cross-encoder (pairwise) | Best | ~minutes | Impossible |

Going from 100% to 95% recall buys you **100-1000x speedup**.

## 0. Setup

In [ ]:
!pip install faiss-cpu matplotlib pandas seaborn -q

In [ ]:
import numpy as np
import faiss
import time
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_style('whitegrid')
print(f'FAISS version: {faiss.__version__}')
print(f'CPU threads: {faiss.omp_get_max_threads()}')

## 1. Generate a Large-Scale Embedding Dataset

We simulate a realistic scenario: a catalog of items with dense embeddings (e.g., from a Two-Tower model or sentence-BERT). Dimensionality 128 is typical for production retrieval systems.

In [ ]:
DIM = 128  # embedding dimension (typical for production)
N_ITEMS = 2_000_000  # 2 million items
N_QUERIES = 500  # number of test queries

print(f'Generating {N_ITEMS:,} item embeddings ({DIM}d)...')
np.random.seed(42)

# Simulate clustered embeddings (more realistic than pure random)
n_clusters = 200
centroids = np.random.randn(n_clusters, DIM).astype(np.float32)
centroids /= np.linalg.norm(centroids, axis=1, keepdims=True)

# Each item belongs to a cluster + noise
cluster_ids = np.random.randint(0, n_clusters, N_ITEMS)
item_embeddings = centroids[cluster_ids] + np.random.randn(N_ITEMS, DIM).astype(np.float32) * 0.3
# Normalize
item_embeddings /= np.linalg.norm(item_embeddings, axis=1, keepdims=True)
item_embeddings = item_embeddings.astype(np.float32)

# Generate query embeddings (from same distribution)
q_clusters = np.random.randint(0, n_clusters, N_QUERIES)
query_embeddings = centroids[q_clusters] + np.random.randn(N_QUERIES, DIM).astype(np.float32) * 0.2
query_embeddings /= np.linalg.norm(query_embeddings, axis=1, keepdims=True)
query_embeddings = query_embeddings.astype(np.float32)

mem_mb = item_embeddings.nbytes / 1e6
print(f'Item embeddings: {item_embeddings.shape} ({mem_mb:.1f} MB)')
print(f'Query embeddings: {query_embeddings.shape}')

## 2. Feel the Pain: Brute Force at Scale

Let's measure how long exact nearest-neighbor search takes as the catalog grows.

In [ ]:
K = 10  # retrieve top-K items

scales = [10_000, 50_000, 100_000, 500_000, 1_000_000, N_ITEMS]
brute_times = []

for n in scales:
    db = item_embeddings[:n]
    index = faiss.IndexFlatIP(DIM)  # exact inner product search
    index.add(db)
    
    t0 = time.time()
    D, I = index.search(query_embeddings, K)
    elapsed = time.time() - t0
    
    ms_per_query = elapsed / N_QUERIES * 1000
    qps = N_QUERIES / elapsed
    brute_times.append({'n_items': n, 'ms_per_query': ms_per_query, 'qps': qps})
    print(f'  {n:>10,} items: {ms_per_query:7.2f} ms/query ({qps:,.0f} QPS)')

# Also compute ground truth for full dataset (for recall measurement later)
print('\nComputing exact ground truth on full dataset...')
index_exact = faiss.IndexFlatIP(DIM)
index_exact.add(item_embeddings)
D_gt, I_gt = index_exact.search(query_embeddings, K)
print(f'Ground truth ready. Shape: {I_gt.shape}')

In [ ]:
bt = pd.DataFrame(brute_times)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(bt.n_items, bt.ms_per_query, 'o-', color='#e74c3c', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of items in catalog')
axes[0].set_ylabel('Latency (ms per query)')
axes[0].set_title('Brute Force: Latency Scales Linearly')
axes[0].axhline(y=50, color='gray', linestyle='--', label='50ms budget')
axes[0].axhline(y=10, color='gray', linestyle=':', label='10ms budget')
axes[0].legend()
axes[0].set_xscale('log')

axes[1].plot(bt.n_items, bt.qps, 's-', color='steelblue', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of items')
axes[1].set_ylabel('Queries Per Second')
axes[1].set_title('Brute Force: Throughput Drops')
axes[1].set_xscale('log')
axes[1].set_yscale('log')

plt.suptitle(f'Brute Force Exact Search ({DIM}d embeddings, top-{K})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nAt {N_ITEMS:,} items, brute force takes {bt.iloc[-1].ms_per_query:.1f}ms per query.')
print('For a 10ms latency budget, we need something faster!')

## 3. FAISS Index Types — From Exact to Approximate

FAISS provides a spectrum of index types, trading accuracy for speed:

| Index | How it works | Speed | Memory | Recall |
|-------|-------------|-------|--------|--------|
| **Flat** | Brute force, scan all vectors | Slow | Full | 100% |
| **IVF** | Partition into clusters, only search nearby clusters | Fast | Full | ~95% |
| **IVF-PQ** | IVF + compress vectors with Product Quantization | Very fast | ~1/32 | ~90% |
| **HNSW** | Hierarchical graph, navigate to nearest neighbors | Fast | Full + graph | ~98% |

In [ ]:
def measure_index(index_name, index, queries, gt_indices, k=10, n_repeat=3):
    """Measure recall@K and QPS for an index."""
    # Warmup
    index.search(queries[:10], k)
    
    # Measure
    best_time = float('inf')
    D, I = None, None
    for _ in range(n_repeat):
        t0 = time.time()
        D, I = index.search(queries, k)
        best_time = min(best_time, time.time() - t0)
    
    # Recall@K: fraction of true top-K that the index found
    recalls = []
    for i in range(len(queries)):
        gt_set = set(gt_indices[i])
        found = set(I[i])
        recalls.append(len(gt_set & found) / k)
    
    qps = len(queries) / best_time
    return {
        'name': index_name,
        f'Recall@{k}': np.mean(recalls),
        'QPS': qps,
        'ms/query': best_time / len(queries) * 1000,
        'index_size_MB': index.ntotal * DIM * 4 / 1e6 if not hasattr(index, 'sa_code_size') else 'compressed',
    }


results = []

# --- Flat (exact baseline) ---
print('Building Flat index (exact)...')
idx_flat = faiss.IndexFlatIP(DIM)
idx_flat.add(item_embeddings)
r = measure_index('Flat (exact)', idx_flat, query_embeddings, I_gt)
results.append(r)
print(f'  Recall@{K}: {r[f"Recall@{K}"]:.4f}, QPS: {r["QPS"]:,.0f}, {r["ms/query"]:.2f} ms/q')

In [ ]:
# --- IVF (Inverted File Index) ---
# Partition space into nlist clusters, only search nprobe nearest clusters
nlist = 256
print(f'Building IVF index (nlist={nlist})...')
quantizer = faiss.IndexFlatIP(DIM)
idx_ivf = faiss.IndexIVFFlat(quantizer, DIM, nlist, faiss.METRIC_INNER_PRODUCT)
idx_ivf.train(item_embeddings)
idx_ivf.add(item_embeddings)

for nprobe in [1, 4, 16, 32, 64]:
    idx_ivf.nprobe = nprobe
    r = measure_index(f'IVF (nprobe={nprobe})', idx_ivf, query_embeddings, I_gt)
    results.append(r)
    print(f'  nprobe={nprobe:2d}: Recall@{K}={r[f"Recall@{K}"]:.4f}, QPS={r["QPS"]:>7,.0f}, {r["ms/query"]:.2f} ms/q')

In [ ]:
# --- IVF-PQ (IVF + Product Quantization) ---
# Compresses each vector to m bytes. Huge memory savings!
m_pq = 32  # number of sub-quantizers (each encodes DIM/m dimensions into 8 bits)
print(f'Building IVF-PQ index (nlist={nlist}, m={m_pq})...')
idx_ivfpq = faiss.IndexIVFPQ(quantizer, DIM, nlist, m_pq, 8, faiss.METRIC_INNER_PRODUCT)
idx_ivfpq.train(item_embeddings)
idx_ivfpq.add(item_embeddings)

original_mb = item_embeddings.nbytes / 1e6
compressed_mb = N_ITEMS * m_pq / 1e6  # m bytes per vector
print(f'Memory: {original_mb:.0f} MB (original) -> {compressed_mb:.0f} MB (PQ compressed) = {compressed_mb/original_mb:.1%}')

for nprobe in [1, 4, 16, 32, 64]:
    idx_ivfpq.nprobe = nprobe
    r = measure_index(f'IVF-PQ (nprobe={nprobe})', idx_ivfpq, query_embeddings, I_gt)
    results.append(r)
    print(f'  nprobe={nprobe:2d}: Recall@{K}={r[f"Recall@{K}"]:.4f}, QPS={r["QPS"]:>7,.0f}, {r["ms/query"]:.2f} ms/q')

In [ ]:
# --- HNSW (Hierarchical Navigable Small World) ---
# Graph-based index, excellent recall-speed tradeoff
print('Building HNSW index (M=32)...')
idx_hnsw = faiss.IndexHNSWFlat(DIM, 32, faiss.METRIC_INNER_PRODUCT)
idx_hnsw.hnsw.efConstruction = 80
idx_hnsw.add(item_embeddings)

for ef_search in [16, 32, 64, 128, 256]:
    idx_hnsw.hnsw.efSearch = ef_search
    r = measure_index(f'HNSW (efSearch={ef_search})', idx_hnsw, query_embeddings, I_gt)
    results.append(r)
    print(f'  efSearch={ef_search:3d}: Recall@{K}={r[f"Recall@{K}"]:.4f}, QPS={r["QPS"]:>7,.0f}, {r["ms/query"]:.2f} ms/q')

## 4. The Money Plot: Recall vs Speed

In [ ]:
df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(10, 6))

# Group by method type
styles = {'Flat': ('X', '#e74c3c', 200), 'IVF ': ('o', '#3498db', 100),
          'IVF-PQ': ('s', '#27ae60', 100), 'HNSW': ('D', '#9b59b6', 100)}

for prefix, (marker, color, ms) in styles.items():
    mask = df['name'].str.startswith(prefix)
    subset = df[mask]
    if len(subset) == 0:
        continue
    ax.scatter(subset['QPS'], subset[f'Recall@{K}'],
               marker=marker, c=color, s=ms, label=prefix.strip(), zorder=5)
    if len(subset) > 1:
        ax.plot(subset['QPS'], subset[f'Recall@{K}'],
                color=color, linewidth=1.5, alpha=0.5)

ax.set_xscale('log')
ax.set_xlabel('Queries Per Second (higher = faster)', fontsize=12)
ax.set_ylabel(f'Recall@{K} (higher = more accurate)', fontsize=12)
ax.set_title(f'Retrieval Tradeoff: Speed vs Accuracy ({N_ITEMS:,} items, {DIM}d)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.axhline(y=0.95, color='gray', linestyle=':', alpha=0.5, label='95% recall')
ax.set_ylim(0.4, 1.02)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\n--- Full Results Table ---')
print(df[['name', f'Recall@{K}', 'QPS', 'ms/query']].to_string(index=False, float_format='%.4f'))

## 5. The Production Cascade

Why not just use the most accurate (but slow) model everywhere?

```
2,000,000 items
    │
    ▼  Stage 1: ANN Retrieval (IVF-PQ, ~1ms)
  1,000 candidates
    │
    ▼  Stage 2: Neural re-ranker (~10ms)
    100 candidates  
    │
    ▼  Stage 3: Business rules / diversity (~1ms)
    10 results → shown to user
```

**Total latency: ~12ms** for searching through 2M items!

Let's simulate this cascade and compare to single-stage approaches.

In [ ]:
# Simulate: how much can a re-ranker recover from a lossy retriever?
# Stage 1: ANN retrieves top-100 candidates (with some recall loss)
# Stage 2: Exact re-ranking of those 100 candidates

def cascade_recall(ann_index, queries, gt_indices, k_retrieve=100, k_final=10):
    """Simulate two-stage cascade: ANN retrieval + exact re-ranking."""
    # Stage 1: ANN retrieves k_retrieve candidates
    t0 = time.time()
    D_ann, I_ann = ann_index.search(queries, k_retrieve)
    stage1_time = time.time() - t0
    
    # Stage 2: exact re-ranking of candidates
    t0 = time.time()
    recalls = []
    for i in range(len(queries)):
        candidates = I_ann[i]
        # Re-score with exact inner product
        candidate_embs = item_embeddings[candidates]
        scores = candidate_embs @ queries[i]
        top_k = candidates[np.argsort(scores)[::-1][:k_final]]
        
        gt_set = set(gt_indices[i])
        recalls.append(len(set(top_k) & gt_set) / k_final)
    stage2_time = time.time() - t0
    
    return {
        f'Recall@{k_final}': np.mean(recalls),
        'Stage1 (ms)': stage1_time / len(queries) * 1000,
        'Stage2 (ms)': stage2_time / len(queries) * 1000,
        'Total (ms)': (stage1_time + stage2_time) / len(queries) * 1000,
    }


# Test cascade with different stage-1 retrievers and budgets
print(f'Cascade: ANN retrieves top-K candidates, then exact re-rank to top-{K}')
print(f'{"":-<70}')

cascade_results = []
idx_ivfpq.nprobe = 8

for k_retrieve in [20, 50, 100, 200, 500, 1000]:
    r = cascade_recall(idx_ivfpq, query_embeddings, I_gt, k_retrieve=k_retrieve, k_final=K)
    r['k_retrieve'] = k_retrieve
    cascade_results.append(r)
    print(f'  Retrieve top-{k_retrieve:>4d} -> rerank -> Recall@{K}={r[f"Recall@{K}"]:.4f}  '
          f'(stage1: {r["Stage1 (ms)"]:.2f}ms, stage2: {r["Stage2 (ms)"]:.2f}ms, '
          f'total: {r["Total (ms)"]:.2f}ms)')

# Compare with exact brute force
print(f'\n  Brute force exact:              Recall@{K}=1.0000  '
      f'({bt.iloc[-1].ms_per_query:.2f}ms)')

In [ ]:
cr = pd.DataFrame(cascade_results)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(cr.k_retrieve, cr[f'Recall@{K}'], 'o-', color='#27ae60', linewidth=2, markersize=8)
axes[0].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Exact (brute force)')
axes[0].set_xlabel('Stage 1: retrieve top-K candidates')
axes[0].set_ylabel(f'Final Recall@{K} (after re-ranking)')
axes[0].set_title('Cascade: More candidates = higher recall')
axes[0].legend()

axes[1].bar(range(len(cr)), cr['Stage1 (ms)'], label='Stage 1 (ANN)', color='#3498db')
axes[1].bar(range(len(cr)), cr['Stage2 (ms)'], bottom=cr['Stage1 (ms)'],
           label='Stage 2 (re-rank)', color='#e74c3c')
axes[1].set_xticks(range(len(cr)))
axes[1].set_xticklabels([f'top-{k}' for k in cr.k_retrieve])
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Cascade: Latency breakdown')
axes[1].legend()
axes[1].axhline(y=bt.iloc[-1].ms_per_query, color='gray', linestyle='--',
               label=f'Brute force ({bt.iloc[-1].ms_per_query:.0f}ms)')
axes[1].legend()

plt.suptitle(f'Two-Stage Cascade on {N_ITEMS:,} Items', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

sweet_spot = cr.iloc[2]  # k_retrieve=100 is usually the sweet spot
print(f'Sweet spot: retrieve top-{int(sweet_spot.k_retrieve)} candidates')
print(f'  Recall@{K}: {sweet_spot[f"Recall@{K}"]:.4f}')
print(f'  Total latency: {sweet_spot["Total (ms)"]:.2f}ms')
print(f'  vs brute force: {bt.iloc[-1].ms_per_query:.2f}ms')
print(f'  Speedup: {bt.iloc[-1].ms_per_query / sweet_spot["Total (ms)"]:.0f}x')

## 6. Interactive Experiment: Index Factory

FAISS has a string-based "index factory" API that makes it easy to try different configurations.

In [ ]:
# Try different index configurations with one-liners
configs = [
    ('Flat', 'Flat'),
    ('IVF256,Flat', 'IVF256,Flat'),
    ('IVF512,Flat', 'IVF512,Flat'),
    ('IVF256,PQ32', 'IVF256,PQ32'),
    ('IVF512,PQ32', 'IVF512,PQ32'),
    ('IVF256,PQ16', 'IVF256,PQ16'),
    ('HNSW32', 'HNSW32'),
]

factory_results = []
for name, factory_str in configs:
    idx = faiss.index_factory(DIM, factory_str, faiss.METRIC_INNER_PRODUCT)
    idx.train(item_embeddings)
    idx.add(item_embeddings)
    
    # Set reasonable search params
    if hasattr(idx, 'nprobe'): idx.nprobe = 16
    if hasattr(idx, 'hnsw'): idx.hnsw.efSearch = 64
    
    r = measure_index(name, idx, query_embeddings, I_gt)
    factory_results.append(r)
    print(f'  {name:20s}: Recall@{K}={r[f"Recall@{K}"]:.4f}, {r["QPS"]:>7,.0f} QPS, {r["ms/query"]:.2f} ms/q')

print('\nUse faiss.index_factory(dim, "IVF256,PQ32", faiss.METRIC_INNER_PRODUCT) to create any of these!')

## Key Takeaways

1. **Brute force scales linearly** — O(n) per query, unusable beyond ~100K items in real-time
2. **ANN gives 100-1000x speedup** at the cost of 2-10% recall loss
3. **IVF-PQ** is the production workhorse: fast search + massive memory compression (32x!)
4. **HNSW** gives the best recall at moderate speed — great when you have enough RAM
5. **The cascade (ANN → re-ranker)** recovers most of the recall loss while staying fast
6. **The tradeoff is tunable**: `nprobe`, `efSearch`, `k_retrieve` let you dial in the sweet spot

### Connection to previous tutorials

| This tutorial | Query Suggestion tutorial | LTR tutorial |
|--------------|--------------------------|-------------|
| ANN retrieval (FAISS) | Two-Tower / GRU models | — |
| Stage 1: fast, approximate | Generates embeddings | — |
| Stage 2: re-ranker | — | LambdaMART ranking |

In a full system: **Two-Tower model** produces embeddings → **FAISS** indexes them → **ANN search** retrieves top-100 → **LambdaMART** re-ranks to final top-10.